# **Dataset Construction**

In [ ]:
# Reload QuAC explicitly to avoid variable collision
with open("train_v0.2.json") as f:
    quac_train = json.load(f)

In [ ]:
# ================================================================
# UNIFIED DATASET CONSTRUCTION
# Decision-Aware LLM Schema
# Sources: QuAC + ShARC + HotpotQA + ContractNLI
# ================================================================

import json
import uuid
import os

output_dir = "/content/unified_dataset"
os.makedirs(output_dir, exist_ok=True)


# ── HELPERS ──────────────────────────────────────────────────────

def make_id(prefix, idx):
    return f"{prefix}_{str(idx).zfill(6)}"

def heuristic_difficulty(num_missing, evidence_len=0, num_hops=0):
    score = num_missing + evidence_len + num_hops
    if score == 0:   return "easy"
    elif score <= 2: return "medium"
    elif score <= 4: return "hard"
    else:            return "very_hard"

def heuristic_failure_mode(action, num_missing, multi_hop=False):
    if action == "ANSWER":             return "COMPLETE"
    if action == "ABSTAIN":            return "INSUFFICIENT_VARIABLES"
    if action == "ASK" and multi_hop:  return "MULTI_HOP_REQUIRED"
    if action == "ASK":                return "INSUFFICIENT_VARIABLES"
    return "COMPLETE"

def heuristic_completeness(action):
    if action == "ANSWER":  return "complete"
    if action == "ASK":     return "partial"
    if action == "ABSTAIN": return "incomplete"
    return "partial"


# ================================================================
# 1. QuAC → ANSWER / ABSTAIN / ASK
# ================================================================
# Action mapping:
#   CANNOTANSWER answer   → ABSTAIN
#   followup = y          → ASK  (after answering)
#   otherwise             → ANSWER

quac_samples = []
sample_idx   = 0

for article in quac_train["data"]:       # train_data from QuAC loading cell
    for para in article["paragraphs"]:
        context_text = para.get("context", "")
        dialogue_id  = para.get("id", str(uuid.uuid4()))

        for turn_id, qa in enumerate(para["qas"]):
            ans_text  = qa["answers"][0]["text"] if qa["answers"] else "CANNOTANSWER"
            followup  = qa.get("followup", "n")
            is_cannot = (ans_text == "CANNOTANSWER")

            # Action
            if is_cannot:
                action   = "ABSTAIN"
                response = "I do not have enough information to answer this question."
            elif followup == "y":
                action   = "ASK"
                response = "Could you provide more details so I can give a more precise answer?"
            else:
                action   = "ANSWER"
                response = ans_text

            num_missing   = 1 if action != "ANSWER" else 0
            failure_mode  = heuristic_failure_mode(action, num_missing)
            difficulty    = heuristic_difficulty(num_missing)
            completeness  = heuristic_completeness(action)

            sample = {
                "id"     : make_id("quac", sample_idx),
                "query"  : qa["question"],
                "context": {
                    "documents": [
                        {"doc_id": dialogue_id, "text": context_text}
                    ]
                },
                "state": {
                    "known_variables"  : [],   # LLM inference later
                    "missing_variables": [],   # LLM inference later
                    "constraints"      : [],   # LLM inference later
                    "failure_mode"     : failure_mode,
                    "difficulty"       : difficulty,
                    "completeness"     : completeness
                },
                "action"  : action,
                "response": response,
                "metadata": {
                    "num_missing_variables": num_missing,
                    "variable_types"       : [],
                    "multi_turn"           : True,
                    "turn_id"              : turn_id + 1,
                    "dialogue_id"          : dialogue_id,
                    "requires_reasoning"   : False,
                    "source"               : "quac",
                    "yesno"                : qa.get("yesno", "x"),
                    "followup_flag"        : followup
                }
            }
            quac_samples.append(sample)
            sample_idx += 1

print(f"QuAC samples constructed : {len(quac_samples)}")


# ================================================================
# 2. ShARC → ANSWER / ASK / ABSTAIN
# ================================================================
# Action mapping:
#   Yes / No     → ANSWER
#   Follow-on    → ASK   (evidence chain gives clarification questions)
#   Irrelevant   → ABSTAIN

sharc_samples = []
sample_idx    = 0

# train_data here is sharc train_data — rename to avoid collision
with open("/content/sharc/sharc1-official/json/sharc_train.json") as f:
    sharc_train = json.load(f)

for item in sharc_train:
    answer   = item["answer"]
    evidence = item.get("evidence", [])
    history  = item.get("history", [])
    scenario = item.get("scenario", "").strip()

    # Action
    if answer in ["Yes", "No"]:
        action   = "ANSWER"
        response = answer
    elif answer == "Follow-on":
        action = "ASK"
        # Use first unanswered follow-up as the response if available
        response = (
            evidence[0]["follow_up_question"]
            if evidence else "Could you provide more details?"
        )
    else:  # Irrelevant
        action   = "ABSTAIN"
        response = "I do not have enough information to determine this."

    num_missing  = len(evidence) if action == "ASK" else (1 if action == "ABSTAIN" else 0)
    failure_mode = heuristic_failure_mode(action, num_missing)
    difficulty   = heuristic_difficulty(num_missing, len(evidence), len(history))
    completeness = heuristic_completeness(action)

    # Build context: snippet + scenario
    context_text = item.get("snippet", "")
    if scenario:
        context_text += f"\n\nUser Scenario: {scenario}"

    # Build missing variables from evidence chain (explicit supervision)
    missing_vars = [ev["follow_up_question"] for ev in evidence]

    sample = {
        "id"     : make_id("sharc", sample_idx),
        "query"  : item["question"],
        "context": {
            "documents": [
                {
                    "doc_id": item["tree_id"],
                    "text"  : context_text,
                    "url"   : item.get("source_url", "")
                }
            ]
        },
        "state": {
            "known_variables"  : [],            # LLM inference later
            "missing_variables": missing_vars,  # directly from evidence chain ✅
            "constraints"      : [],            # LLM inference later
            "failure_mode"     : failure_mode,
            "difficulty"       : difficulty,
            "completeness"     : completeness
        },
        "action"  : action,
        "response": response,
        "metadata": {
            "num_missing_variables": num_missing,
            "variable_types"       : [],
            "multi_turn"           : len(history) > 0,
            "turn_id"              : None,
            "dialogue_id"          : item["tree_id"],
            "requires_reasoning"   : len(evidence) > 1,
            "source"               : "sharc",
            "utterance_id"         : item["utterance_id"],
            "sharc_answer"         : answer,
            "evidence_depth"       : len(evidence),
            "history_depth"        : len(history)
        }
    }
    sharc_samples.append(sample)
    sample_idx += 1

print(f"ShARC samples constructed : {len(sharc_samples)}")


# ================================================================
# 3. HotpotQA → ANSWER only
# ================================================================
# No ABSTAIN/ASK here — used purely for ANSWER supervision
# Difficulty from dataset's own 'level' field

hotpot_samples = []
sample_idx     = 0

# dataset["train"] from HotpotQA loading cell
for item in dataset["train"]:
    sf_titles  = item["supporting_facts"]["title"]
    sf_sentids = item["supporting_facts"]["sent_id"]
    ctx_titles = item["context"]["title"]
    ctx_sents  = item["context"]["sentences"]

    title_to_sents = {t: s for t, s in zip(ctx_titles, ctx_sents)}

    # Build context from supporting facts only
    supporting_texts = []
    for t, sid in zip(sf_titles, sf_sentids):
        if t in title_to_sents and sid < len(title_to_sents[t]):
            supporting_texts.append({
                "doc_id": t,
                "text"  : title_to_sents[t][sid]
            })

    level_map  = {"easy": "easy", "medium": "medium", "hard": "hard"}
    difficulty = level_map.get(item.get("level", "medium"), "medium")

    sample = {
        "id"     : make_id("hotpot", sample_idx),
        "query"  : item["question"],
        "context": {"documents": supporting_texts},
        "state": {
            "known_variables"  : [],       # LLM inference later
            "missing_variables": [],       # always empty — ANSWER action
            "constraints"      : [],
            "failure_mode"     : "COMPLETE",
            "difficulty"       : difficulty,
            "completeness"     : "complete"
        },
        "action"  : "ANSWER",
        "response": item["answer"],
        "metadata": {
            "num_missing_variables": 0,
            "variable_types"       : [],
            "multi_turn"           : False,
            "turn_id"              : None,
            "dialogue_id"          : None,
            "requires_reasoning"   : item.get("type") == "bridge",
            "source"               : "hotpotqa",
            "question_type"        : item.get("type", ""),
            "level"                : item.get("level", ""),
            "num_supporting_facts" : len(sf_titles)
        }
    }
    hotpot_samples.append(sample)
    sample_idx += 1

print(f"HotpotQA samples constructed : {len(hotpot_samples)}")


# ================================================================
# 4. ContractNLI → ANSWER / ABSTAIN
# ================================================================
# Entailment / Contradiction  → ANSWER
# NotMentioned                → ABSTAIN
# hypothesis plays role of query
# response is synthetic template — LLM generation deferred

with open("/content/contract_nli/contract-nli/train.json") as f:
    cnli_data = json.load(f)

cnli_labels    = cnli_data["labels"]
cnli_documents = cnli_data["documents"]
cnli_samples   = []
sample_idx     = 0

for doc in cnli_documents:
    doc_text   = doc["text"]
    char_spans = doc["spans"]

    for annot_set in doc["annotation_sets"]:
        for label_id, annot in annot_set["annotations"].items():
            choice    = annot["choice"]
            span_idxs = annot["spans"]
            hypothesis = cnli_labels[label_id]["hypothesis"]
            short_desc = cnli_labels[label_id]["short_description"]

            # Resolve supporting span texts
            span_texts = []
            for idx in span_idxs:
                if idx < len(char_spans):
                    s, e = char_spans[idx]
                    span_texts.append(doc_text[s:e].strip())

            # Action
            if choice in ["Entailment", "Contradiction"]:
                action = "ANSWER"
                if choice == "Entailment":
                    response = f"Yes, the contract supports: {short_desc}."
                else:
                    response = f"No, the contract contradicts: {short_desc}."
            else:  # NotMentioned
                action   = "ABSTAIN"
                response = f"The contract does not mention information related to: {short_desc}."

            num_missing  = 0 if action == "ANSWER" else 1
            failure_mode = heuristic_failure_mode(action, num_missing)
            difficulty   = heuristic_difficulty(num_missing, num_hops=len(span_idxs))
            completeness = heuristic_completeness(action)

            sample = {
                "id"     : make_id("cnli", sample_idx),
                "query"  : hypothesis,          # hypothesis as query ✅
                "context": {
                    "documents": [
                        {
                            "doc_id"   : str(doc["id"]),
                            "file_name": doc.get("file_name", ""),
                            "text"     : doc_text,
                            "spans"    : span_texts  # resolved supporting text
                        }
                    ]
                },
                "state": {
                    "known_variables"  : [short_desc],  # clause type known ✅
                    "missing_variables": [] if action == "ANSWER" else [short_desc],
                    "constraints"      : [],             # LLM inference later
                    "failure_mode"     : failure_mode,
                    "difficulty"       : difficulty,
                    "completeness"     : completeness
                },
                "action"  : action,
                "response": response,           # template — LLM to refine later
                "metadata": {
                    "num_missing_variables": num_missing,
                    "variable_types"       : ["constraint"],
                    "multi_turn"           : False,
                    "turn_id"              : None,
                    "dialogue_id"          : None,
                    "requires_reasoning"   : len(span_idxs) > 1,
                    "source"               : "contract_nli",
                    "label_id"             : label_id,
                    "nli_choice"           : choice,
                    "num_spans"            : len(span_idxs)
                }
            }
            cnli_samples.append(sample)
            sample_idx += 1

print(f"ContractNLI samples constructed : {len(cnli_samples)}")


# ================================================================
# MERGE + SAVE
# ================================================================

all_samples = quac_samples + sharc_samples + hotpot_samples + cnli_samples

# Action distribution check
from collections import Counter
action_counts = Counter(s["action"] for s in all_samples)
source_counts = Counter(s["metadata"]["source"] for s in all_samples)

print("\n" + "="*50)
print(f"Total samples : {len(all_samples)}")
print(f"\nAction distribution:")
for k, v in action_counts.items():
    print(f"  {k:10} : {v:6}  ({100*v/len(all_samples):.1f}%)")
print(f"\nSource distribution:")
for k, v in source_counts.items():
    print(f"  {k:15} : {v}")

# Save full dataset
out_path = f"{output_dir}/unified_train.jsonl"
with open(out_path, "w") as f:
    for sample in all_samples:
        f.write(json.dumps(sample) + "\n")

print(f"\nSaved → {out_path}")

# Save per-source splits too (useful for ablations)
for source_name, source_samples in [
    ("quac", quac_samples),
    ("sharc", sharc_samples),
    ("hotpotqa", hotpot_samples),
    ("contract_nli", cnli_samples)
]:
    path = f"{output_dir}/{source_name}_train.jsonl"
    with open(path, "w") as f:
        for s in source_samples:
            f.write(json.dumps(s) + "\n")
    print(f"Saved {source_name:15} → {path}")

In [ ]:
# Save as JSON too (list format)
json_out_path = f"{output_dir}/unified_train.json"
with open(json_out_path, "w") as f:
    json.dump(all_samples, f, indent=2)
print(f"Saved JSON  → {json_out_path}")

# Per-source JSON files too
for source_name, source_samples in [
    ("quac", quac_samples),
    ("sharc", sharc_samples),
    ("hotpotqa", hotpot_samples),
    ("contract_nli", cnli_samples)
]:
    path = f"{output_dir}/{source_name}_train.json"
    with open(path, "w") as f:
        json.dump(source_samples, f, indent=2)
    print(f"Saved {source_name:15} → {path}")

In [ ]:
# ================================================================
# PART 1 — Raw JSON preview: 1 sample per source
# ================================================================

import json
from collections import defaultdict

# Load all samples by source
source_samples = defaultdict(list)
with open("/content/unified_dataset/unified_train.jsonl") as f:
    for line in f:
        s = json.loads(line)
        source_samples[s["metadata"]["source"]].append(s)

# Print 1 raw sample per source
for source in ["quac", "sharc", "hotpotqa", "contract_nli"]:
    print("\n" + "="*70)
    print(f"SOURCE: {source.upper()}")
    print("="*70)
    print(json.dumps(source_samples[source][0], indent=2))

In [ ]:
# ================================================================
# PART 2 — Dynamic interactive viewer (ipywidgets)
# ================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json
from collections import defaultdict

# ── Group by source → then by context (doc_id or dialogue_id) ──

def get_context_key(sample):
    src = sample["metadata"]["source"]
    if src == "quac":
        return sample["metadata"].get("dialogue_id", "unknown")
    elif src == "sharc":
        return sample["metadata"].get("utterance_id", sample["id"])
    elif src == "hotpotqa":
        return sample["id"]   # each is standalone
    elif src == "contract_nli":
        docs = sample["context"]["documents"]
        return docs[0]["doc_id"] if docs else "unknown"
    return "unknown"

def get_context_text(sample):
    docs = sample["context"]["documents"]
    if not docs:
        return "— no context —"
    return docs[0]["text"][:600].replace("\n", " ").strip() + "..."

# Build nested dict: source → context_key → [samples]
grouped = defaultdict(lambda: defaultdict(list))

with open("/content/unified_dataset/unified_train.jsonl") as f:
    for line in f:
        s = json.loads(line)
        src = s["metadata"]["source"]
        ctx_key = get_context_key(s)
        grouped[src][ctx_key].append(s)

sources = ["quac", "sharc", "hotpotqa", "contract_nli"]

# ── Widgets ───────────────────────────────────────────────────

source_dropdown = widgets.Dropdown(
    options=sources,
    description="📂 Dataset:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px")
)

context_dropdown = widgets.Dropdown(
    description="📄 Context ID:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="500px")
)

sample_slider = widgets.IntSlider(
    value=0, min=0, max=0, step=1,
    description="🔢 Sample:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="400px")
)

view_toggle = widgets.ToggleButtons(
    options=["Formatted View", "Raw JSON"],
    description="👁 View:",
    style={"description_width": "80px", "button_width": "140px"}
)

out = widgets.Output()

# ── Formatted renderer ────────────────────────────────────────

ACTION_COLOR = {"ANSWER": "#2e7d32", "ASK": "#1565c0", "ABSTAIN": "#b71c1c"}

def render_formatted(sample):
    action      = sample["action"]
    ACTION_COLOR = {"ANSWER": "#00e676", "ASK": "#40c4ff", "ABSTAIN": "#ff5252"}
    color       = ACTION_COLOR.get(action, "#ffffff")
    state       = sample["state"]
    meta        = sample["metadata"]
    ctx_text    = get_context_text(sample)
    missing     = state.get("missing_variables", [])
    missing_str = "<br>".join(f"• {m}" for m in missing) if missing else "— none —"

    html = f"""
    <div style="font-family:monospace; font-size:13px; padding:14px;
                border:1px solid #444; border-radius:8px;
                background:#1e1e1e; color:#e0e0e0;">

      <div style="margin-bottom:8px; color:#bbbbbb;">
        <b style="color:#ffffff;">🆔 ID:</b> {sample['id']}
        &nbsp;|&nbsp;
        <b style="color:#ffffff;">Source:</b> {meta['source']}
        &nbsp;|&nbsp;
        <b style="color:#ffffff;">Multi-turn:</b> {meta.get('multi_turn', False)}
        &nbsp;|&nbsp;
        <b style="color:#ffffff;">Turn:</b> {meta.get('turn_id', '—')}
      </div>

      <hr style="border:0.5px solid #444;">

      <div style="margin-bottom:10px;">
        <b style="color:#90caf9;">📄 CONTEXT</b><br>
        <div style="background:#263238; padding:10px; border-radius:4px;
                    font-size:12px; white-space:pre-wrap;
                    color:#cfd8dc; margin-top:4px;">
          {ctx_text}
        </div>
      </div>

      <div style="margin-bottom:10px;">
        <b style="color:#90caf9;">❓ QUERY:</b>
        <span style="color:#fff176;"> {sample['query']}</span>
      </div>

      <hr style="border:0.5px solid #444;">

      <table style="width:100%; font-size:12px; border-collapse:collapse; color:#e0e0e0;">
        <tr>
          <td style="padding:6px; width:50%; vertical-align:top;">
            <b style="color:#ce93d8;">🧠 STATE</b><br><br>
            <b>Failure mode:</b> <span style="color:#ffcc80;">{state.get('failure_mode','—')}</span><br>
            <b>Difficulty:</b>   <span style="color:#ffcc80;">{state.get('difficulty','—')}</span><br>
            <b>Completeness:</b> <span style="color:#ffcc80;">{state.get('completeness','—')}</span><br>
            <b>Missing vars:</b><br>
            <span style="color:#ef9a9a;">{missing_str}</span>
          </td>
          <td style="padding:6px; width:50%; vertical-align:top;">
            <b style="color:#ce93d8;">📊 METADATA</b><br><br>
            <b>Requires reasoning:</b> <span style="color:#ffcc80;">{meta.get('requires_reasoning','—')}</span><br>
            <b># Missing vars:</b>     <span style="color:#ffcc80;">{meta.get('num_missing_variables','—')}</span><br>
            {"<b>Evidence depth:</b> <span style='color:#ffcc80;'>" + str(meta.get('evidence_depth','—')) + "</span><br>"
              if 'evidence_depth' in meta else ""}
            {"<b>NLI choice:</b> <span style='color:#ffcc80;'>"     + str(meta.get('nli_choice','—'))     + "</span><br>"
              if 'nli_choice' in meta else ""}
            {"<b>Question type:</b> <span style='color:#ffcc80;'>"  + str(meta.get('question_type','—'))  + "</span><br>"
              if 'question_type' in meta else ""}
          </td>
        </tr>
      </table>

      <hr style="border:0.5px solid #444;">

      <div style="margin-top:8px;">
        <b style="color:#90caf9;">🎯 ACTION:</b>
        <span style="color:{color}; font-weight:bold; font-size:15px;">
          ● {action}
        </span>
      </div>

      <div style="margin-top:8px;">
        <b style="color:#90caf9;">💬 RESPONSE:</b>
        <div style="background:#1b3a2b; padding:10px; border-radius:4px;
                    font-size:12px; border-left:3px solid {color};
                    color:#c8e6c9; margin-top:4px;">
          {sample['response']}
        </div>
      </div>

    </div>
    """
    return html


# ── Update functions ──────────────────────────────────────────

def update_context_dropdown(change=None):
    src  = source_dropdown.value
    keys = list(grouped[src].keys())
    context_dropdown.options = keys[:200]   # cap at 200 for performance
    context_dropdown.value   = keys[0] if keys else None

def update_slider(change=None):
    src     = source_dropdown.value
    ctx_key = context_dropdown.value
    samples = grouped[src].get(ctx_key, [])
    sample_slider.max   = max(0, len(samples) - 1)
    sample_slider.value = 0

def render(change=None):
    src     = source_dropdown.value
    ctx_key = context_dropdown.value
    samples = grouped[src].get(ctx_key, [])
    idx     = sample_slider.value

    with out:
        clear_output(wait=True)
        if not samples:
            print("No samples found.")
            return

        print(f"Context: '{ctx_key[:60]}...' | {len(samples)} turns | "
              f"Highlighted: turn {idx+1}  [{src}]")

        if view_toggle.value == "Raw JSON":
            for i, s in enumerate(samples):
                marker = "  ◀ SELECTED" if i == idx else ""
                print(f"\n{'='*60}")
                print(f"TURN {i+1}{marker}")
                print('='*60)
                print(json.dumps(s, indent=2))
        else:
            # Show all turns, highlight selected one
            all_html = ""
            for i, s in enumerate(samples):
                border = "3px solid #00e676" if i == idx else "1px solid #444"
                opacity = "1.0" if i == idx else "0.55"
                all_html += f"""
                <div style="margin-bottom:12px; opacity:{opacity};
                            border:{border}; border-radius:8px;">
                  <div style="background:#2a2a2a; padding:4px 10px;
                              border-radius:6px 6px 0 0; color:#aaa;
                              font-size:11px; font-family:monospace;">
                    TURN {i+1} of {len(samples)}
                    {"&nbsp;◀ <b style='color:#00e676;'>SELECTED</b>" if i == idx else ""}
                  </div>
                  {render_formatted(s)}
                </div>
                """
            display(HTML(all_html))

# ── Wire up observers ─────────────────────────────────────────

source_dropdown.observe(update_context_dropdown, names="value")
source_dropdown.observe(update_slider,           names="value")
context_dropdown.observe(update_slider,          names="value")
context_dropdown.observe(render,                 names="value")
sample_slider.observe(render,                    names="value")
view_toggle.observe(render,                      names="value")

# ── Initial render ────────────────────────────────────────────

update_context_dropdown()
update_slider()

display(widgets.VBox([
    widgets.HBox([source_dropdown, context_dropdown]),
    widgets.HBox([sample_slider, view_toggle]),
    out
]))

render()

In [ ]:
# ================================================================
# UNIFIED DATASET — FULL EDA + SAMPLE VIEWER
# ================================================================

import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter, defaultdict
import numpy as np

# ── Load ─────────────────────────────────────────────────────
with open("/content/unified_dataset/unified_train.json") as f:
    all_samples = json.load(f)

print(f"Total samples: {len(all_samples)}")


# ── Helper ───────────────────────────────────────────────────
def get_ctx_text(sample):
    docs = sample["context"]["documents"]
    return " ".join(d["text"].strip() for d in docs if d.get("text","").strip())

def hash_text(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def sep(title="", w=65):
    print("\n" + "="*w)
    if title:
        print(f"  {title}")
        print("="*w)

def show_sample(s, ctx_chars=300, indent="  "):
    ctx = get_ctx_text(s).replace("\n"," ")[:ctx_chars]
    print(f"{indent}ID       : {s['id']}")
    print(f"{indent}Source   : {s['metadata']['source']}")
    print(f"{indent}CONTEXT  : {ctx}...")
    print(f"{indent}QUERY    : {s['query']}")
    print(f"{indent}ACTION   : {s['action']}")
    print(f"{indent}RESPONSE : {s['response'][:150]}")
    print(f"{indent}State    : failure={s['state']['failure_mode']} | "
          f"difficulty={s['state']['difficulty']} | "
          f"completeness={s['state']['completeness']}")
    print(f"{indent}Meta     : multi_turn={s['metadata']['multi_turn']} | "
          f"turn_id={s['metadata'].get('turn_id','—')} | "
          f"requires_reasoning={s['metadata'].get('requires_reasoning','—')}")


# ── Build dataframe ───────────────────────────────────────────
records = []
for s in all_samples:
    ctx_text = get_ctx_text(s)
    records.append({
        "id"                   : s["id"],
        "source"               : s["metadata"]["source"],
        "action"               : s["action"],
        "difficulty"           : s["state"]["difficulty"],
        "failure_mode"         : s["state"]["failure_mode"],
        "completeness"         : s["state"]["completeness"],
        "multi_turn"           : s["metadata"].get("multi_turn", False),
        "turn_id"              : s["metadata"].get("turn_id", None),
        "dialogue_id"          : s["metadata"].get("dialogue_id", None),
        "requires_reasoning"   : s["metadata"].get("requires_reasoning", False),
        "num_missing"          : s["metadata"].get("num_missing_variables", 0),
        "ctx_hash"             : hash_text(ctx_text),
        "ctx_len"              : len(ctx_text.split()),
        "q_len"                : len(s["query"].split()),
        "r_len"                : len(s["response"].split()),
        "_sample"              : s   # keep reference
    })

df = pd.DataFrame(records)


# ================================================================
# 1. ACTION DISTRIBUTION
# ================================================================
sep("1. ACTION DISTRIBUTION (ANSWER / ASK / ABSTAIN)")

action_counts = df["action"].value_counts()
total = len(df)
for act, cnt in action_counts.items():
    print(f"  {act:10} : {cnt:6}  ({100*cnt/total:.1f}%)")

# Samples
for act in ["ANSWER", "ASK", "ABSTAIN"]:
    sep(f"SAMPLE — ACTION = {act}", w=55)
    subset = df[df["action"] == act]
    for _, row in subset.sample(min(2, len(subset)), random_state=42).iterrows():
        show_sample(row["_sample"])
        print()


# ================================================================
# 2. DIFFICULTY DISTRIBUTION
# ================================================================
sep("2. DIFFICULTY DISTRIBUTION")

diff_counts = df["difficulty"].value_counts()
for d, cnt in diff_counts.items():
    print(f"  {d:12} : {cnt:6}  ({100*cnt/total:.1f}%)")

# Cross tab with action
print("\n  --- Difficulty × Action ---")
print(pd.crosstab(df["difficulty"], df["action"]).to_string())

# Samples per difficulty
for diff in ["easy", "medium", "hard", "very_hard"]:
    subset = df[df["difficulty"] == diff]
    if len(subset) == 0:
        continue
    sep(f"SAMPLE — DIFFICULTY = {diff}", w=55)
    for _, row in subset.sample(min(2, len(subset)), random_state=42).iterrows():
        show_sample(row["_sample"])
        print()


# ================================================================
# 3. FAILURE MODE DISTRIBUTION
# ================================================================
sep("3. FAILURE MODE DISTRIBUTION")

fm_counts = df["failure_mode"].value_counts()
for fm, cnt in fm_counts.items():
    print(f"  {fm:30} : {cnt:6}  ({100*cnt/total:.1f}%)")

print("\n  --- Failure Mode × Action ---")
print(pd.crosstab(df["failure_mode"], df["action"]).to_string())

# Samples
for fm in df["failure_mode"].unique():
    subset = df[df["failure_mode"] == fm]
    sep(f"SAMPLE — FAILURE MODE = {fm}", w=55)
    for _, row in subset.sample(min(2, len(subset)), random_state=42).iterrows():
        show_sample(row["_sample"])
        print()


# ================================================================
# 4. UNIQUE CONTEXTS (DEDUPLICATION)
# ================================================================
sep("4. UNIQUE CONTEXT ANALYSIS")

total_unique_ctx  = df["ctx_hash"].nunique()
total_samples     = len(df)
print(f"  Total samples            : {total_samples}")
print(f"  Unique contexts (hashed) : {total_unique_ctx}")
print(f"  Avg samples/context      : {total_samples/total_unique_ctx:.2f}")

# Per source
print("\n  --- Unique contexts per source ---")
src_ctx = df.groupby("source")["ctx_hash"].nunique()
src_tot = df.groupby("source")["id"].count()
for src in src_ctx.index:
    print(f"  {src:15} : {src_ctx[src]:5} unique contexts | "
          f"{src_tot[src]:6} samples | "
          f"avg {src_tot[src]/src_ctx[src]:.2f} samples/ctx")


# ================================================================
# 5. QUESTIONS PER UNIQUE CONTEXT (min/max/distribution)
# ================================================================
sep("5. QUESTIONS LINKED PER UNIQUE CONTEXT")

ctx_sample_counts = df.groupby("ctx_hash")["id"].count()
print(f"  Min questions/context : {ctx_sample_counts.min()}")
print(f"  Max questions/context : {ctx_sample_counts.max()}")
print(f"  Mean                  : {ctx_sample_counts.mean():.2f}")
print(f"  Median                : {ctx_sample_counts.median():.1f}")
print(f"\n  --- Distribution of #questions per context ---")
dist = ctx_sample_counts.value_counts().sort_index()
for n_q, cnt in dist.items():
    bar = "█" * min(cnt // max(1, dist.max()//30), 30)
    print(f"  {n_q:3} qs → {cnt:5} contexts  {bar}")

# Show context with MAX questions
max_ctx_hash = ctx_sample_counts.idxmax()
max_ctx_df   = df[df["ctx_hash"] == max_ctx_hash]
sep(f"CONTEXT WITH MOST QUESTIONS ({ctx_sample_counts.max()} qs)", w=55)
ctx_preview = get_ctx_text(max_ctx_df.iloc[0]["_sample"]).replace("\n"," ")[:300]
print(f"  Source  : {max_ctx_df.iloc[0]['source']}")
print(f"  CONTEXT : {ctx_preview}...")
print(f"\n  All {len(max_ctx_df)} questions on this context:")
for i, (_, row) in enumerate(max_ctx_df.iterrows(), 1):
    print(f"\n    Q{i}: {row['_sample']['query']}")
    print(f"       ACTION   : {row['action']}")
    print(f"       RESPONSE : {row['_sample']['response'][:120]}")

# Show context with MIN questions > 1
min_ctx_hash = ctx_sample_counts[ctx_sample_counts > 1].idxmin()
min_ctx_df   = df[df["ctx_hash"] == min_ctx_hash]
sep(f"CONTEXT WITH FEWEST QUESTIONS ({ctx_sample_counts[min_ctx_hash]} qs)", w=55)
ctx_preview = get_ctx_text(min_ctx_df.iloc[0]["_sample"]).replace("\n"," ")[:300]
print(f"  Source  : {min_ctx_df.iloc[0]['source']}")
print(f"  CONTEXT : {ctx_preview}...")
for i, (_, row) in enumerate(min_ctx_df.iterrows(), 1):
    print(f"\n    Q{i}: {row['_sample']['query']}")
    print(f"       ACTION   : {row['action']}")
    print(f"       RESPONSE : {row['_sample']['response'][:120]}")


# ================================================================
# 6. MULTI-TURN vs SINGLE-TURN
# ================================================================
sep("6. MULTI-TURN vs SINGLE-TURN ANALYSIS")

mt_counts = df["multi_turn"].value_counts()
print(f"  Multi-turn  : {mt_counts.get(True,0):6}  ({100*mt_counts.get(True,0)/total:.1f}%)")
print(f"  Single-turn : {mt_counts.get(False,0):6}  ({100*mt_counts.get(False,0)/total:.1f}%)")

# Turn depth distribution (for multi-turn)
mt_df = df[df["multi_turn"] == True]
if len(mt_df) > 0 and mt_df["dialogue_id"].notna().any():
    turn_depths = mt_df.groupby("dialogue_id")["turn_id"].max().dropna()
    print(f"\n  --- Turn depth distribution (multi-turn dialogues) ---")
    print(f"  Min turns/dialogue : {turn_depths.min():.0f}")
    print(f"  Max turns/dialogue : {turn_depths.max():.0f}")
    print(f"  Avg turns/dialogue : {turn_depths.mean():.2f}")

    depth_dist = turn_depths.value_counts().sort_index()
    print(f"\n  Turn depth breakdown:")
    for depth, cnt in depth_dist.items():
        bar = "█" * min(int(cnt/max(depth_dist))*20, 20)
        print(f"    {int(depth):2} turns → {cnt:5} dialogues  {bar}")

# Per source multi-turn breakdown
print("\n  --- Multi-turn by source ---")
print(pd.crosstab(df["source"], df["multi_turn"],
                  colnames=["multi_turn"]).to_string())

# Sample — single turn
sep("SAMPLE — SINGLE TURN", w=55)
st_subset = df[df["multi_turn"] == False]
for _, row in st_subset.sample(min(2, len(st_subset)), random_state=42).iterrows():
    show_sample(row["_sample"])
    print()

# Sample — multi turn: show a full dialogue
sep("SAMPLE — MULTI TURN (full dialogue)", w=55)
if len(mt_df) > 0 and mt_df["dialogue_id"].notna().any():
    # Pick a dialogue with 4-6 turns
    good_dialogues = turn_depths[(turn_depths >= 4) & (turn_depths <= 6)]
    if len(good_dialogues) > 0:
        sample_dlg_id = good_dialogues.sample(1, random_state=42).index[0]
        dlg_df = mt_df[mt_df["dialogue_id"] == sample_dlg_id].sort_values("turn_id")
        ctx_preview = get_ctx_text(dlg_df.iloc[0]["_sample"]).replace("\n"," ")[:300]
        print(f"  Dialogue ID : {sample_dlg_id[:60]}")
        print(f"  Source      : {dlg_df.iloc[0]['source']}")
        print(f"  CONTEXT     : {ctx_preview}...")
        print(f"\n  {len(dlg_df)} turns:")
        for _, row in dlg_df.iterrows():
            s = row["_sample"]
            print(f"\n    Turn {row['turn_id']}:")
            print(f"      Q        : {s['query']}")
            print(f"      ACTION   : {s['action']}")
            print(f"      RESPONSE : {s['response'][:120]}")


# ================================================================
# 7. REQUIRES REASONING
# ================================================================
sep("7. REQUIRES REASONING DISTRIBUTION")

rr_counts = df["requires_reasoning"].value_counts()
print(f"  Requires reasoning     : {rr_counts.get(True,0):6}  ({100*rr_counts.get(True,0)/total:.1f}%)")
print(f"  No reasoning required  : {rr_counts.get(False,0):6}  ({100*rr_counts.get(False,0)/total:.1f}%)")

print("\n  --- Requires reasoning by source ---")
print(pd.crosstab(df["source"], df["requires_reasoning"],
                  colnames=["requires_reasoning"]).to_string())

print("\n  --- Requires reasoning × Action ---")
print(pd.crosstab(df["requires_reasoning"], df["action"]).to_string())

# Samples
sep("SAMPLE — REQUIRES REASONING = True", w=55)
rr_sub = df[df["requires_reasoning"] == True]
for _, row in rr_sub.sample(min(2, len(rr_sub)), random_state=42).iterrows():
    show_sample(row["_sample"])
    print()

sep("SAMPLE — REQUIRES REASONING = False", w=55)
nr_sub = df[df["requires_reasoning"] == False]
for _, row in nr_sub.sample(min(2, len(nr_sub)), random_state=42).iterrows():
    show_sample(row["_sample"])
    print()


# ================================================================
# 8. TEXT LENGTH STATS
# ================================================================
sep("8. TEXT LENGTH STATS (words)")

print("  --- Context length ---")
print(df["ctx_len"].describe().round(2).to_string())
print("\n  --- Query length ---")
print(df["q_len"].describe().round(2).to_string())
print("\n  --- Response length ---")
print(df["r_len"].describe().round(2).to_string())

print("\n  --- Avg query length by action ---")
print(df.groupby("action")["q_len"].mean().round(2).to_string())
print("\n  --- Avg response length by action ---")
print(df.groupby("action")["r_len"].mean().round(2).to_string())


# ================================================================
# 9. COMPLETENESS DISTRIBUTION
# ================================================================
sep("9. COMPLETENESS DISTRIBUTION")

comp_counts = df["completeness"].value_counts()
for c, cnt in comp_counts.items():
    print(f"  {c:12} : {cnt:6}  ({100*cnt/total:.1f}%)")

print("\n  --- Completeness × Action ---")
print(pd.crosstab(df["completeness"], df["action"]).to_string())


# ================================================================
# 10. PLOTS — all in one figure
# ================================================================
sep("10. PLOTS")

fig = plt.figure(figsize=(20, 20))
fig.patch.set_facecolor("#1e1e1e")
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {
    "ANSWER" : "#00e676",
    "ASK"    : "#40c4ff",
    "ABSTAIN": "#ff5252",
    "easy"   : "#b9f6ca",
    "medium" : "#fff9c4",
    "hard"   : "#ffccbc",
    "very_hard": "#ef9a9a",
}
BAR_DEFAULTS = ["#80cbc4","#80deea","#b39ddb","#f48fb1","#ffcc80"]

def style_ax(ax, title):
    ax.set_facecolor("#2a2a2a")
    ax.tick_params(colors="#cccccc", labelsize=8)
    ax.title.set_color("#ffffff")
    ax.title.set_fontsize(10)
    ax.set_title(title)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444444")

# 1. Action pie
ax1 = fig.add_subplot(gs[0, 0])
ac  = df["action"].value_counts()
ax1.pie(ac.values, labels=ac.index, autopct="%1.1f%%",
        colors=[COLORS.get(k,"#aaa") for k in ac.index],
        textprops={"color":"white","fontsize":9})
style_ax(ax1, "Action Distribution")

# 2. Difficulty bar
ax2   = fig.add_subplot(gs[0, 1])
dc    = df["difficulty"].value_counts()
order = ["easy","medium","hard","very_hard"]
dc    = dc.reindex([o for o in order if o in dc.index])
ax2.bar(dc.index, dc.values, color=[COLORS.get(k,"#aaa") for k in dc.index])
style_ax(ax2, "Difficulty Distribution")
ax2.set_ylabel("Count", color="#cccccc")

# 3. Failure mode bar
ax3 = fig.add_subplot(gs[0, 2])
fc  = df["failure_mode"].value_counts()
ax3.barh(fc.index, fc.values, color=BAR_DEFAULTS[:len(fc)])
style_ax(ax3, "Failure Mode Distribution")
ax3.tick_params(axis='y', labelsize=7)
ax3.set_xlabel("Count", color="#cccccc")

# 4. Source × Action stacked bar
ax4      = fig.add_subplot(gs[1, 0])
src_act  = pd.crosstab(df["source"], df["action"])
src_act.plot(kind="bar", stacked=True, ax=ax4,
             color=[COLORS.get(c,"#aaa") for c in src_act.columns],
             legend=True)
style_ax(ax4, "Source × Action")
ax4.tick_params(axis='x', rotation=30)
ax4.legend(fontsize=7, facecolor="#333", labelcolor="white")
ax4.set_ylabel("Count", color="#cccccc")

# 5. Questions per unique context histogram
ax5     = fig.add_subplot(gs[1, 1])
ctx_cnt = df.groupby("ctx_hash")["id"].count()
ax5.hist(ctx_cnt.values, bins=30, color="#80cbc4", edgecolor="#1e1e1e")
style_ax(ax5, "Questions per Unique Context")
ax5.set_xlabel("# Questions", color="#cccccc")
ax5.set_ylabel("# Contexts", color="#cccccc")

# 6. Multi-turn pie
ax6  = fig.add_subplot(gs[1, 2])
mt_c = df["multi_turn"].value_counts()
ax6.pie(mt_c.values, labels=["Multi-turn","Single-turn"],
        autopct="%1.1f%%", colors=["#40c4ff","#ffcc80"],
        textprops={"color":"white","fontsize":9})
style_ax(ax6, "Multi-turn vs Single-turn")

# 7. Turn depth histogram (multi-turn only)
ax7 = fig.add_subplot(gs[2, 0])
if len(mt_df) > 0 and mt_df["dialogue_id"].notna().any():
    td = mt_df.groupby("dialogue_id")["turn_id"].max().dropna()
    ax7.hist(td.values, bins=15, color="#ce93d8", edgecolor="#1e1e1e")
style_ax(ax7, "Turn Depth (multi-turn dialogues)")
ax7.set_xlabel("Max turn id", color="#cccccc")
ax7.set_ylabel("# Dialogues", color="#cccccc")

# 8. Requires reasoning by source
ax8  = fig.add_subplot(gs[2, 1])
rr_s = pd.crosstab(df["source"], df["requires_reasoning"])
rr_s.plot(kind="bar", stacked=True, ax=ax8,
          color=["#546e7a","#80cbc4"], legend=True)
style_ax(ax8, "Requires Reasoning by Source")
ax8.tick_params(axis='x', rotation=30)
ax8.legend(["False","True"], fontsize=7, facecolor="#333", labelcolor="white")
ax8.set_ylabel("Count", color="#cccccc")

# 9. Completeness × Action
ax9   = fig.add_subplot(gs[2, 2])
comp  = pd.crosstab(df["completeness"], df["action"])
comp.plot(kind="bar", ax=ax9,
          color=[COLORS.get(c,"#aaa") for c in comp.columns])
style_ax(ax9, "Completeness × Action")
ax9.tick_params(axis='x', rotation=20)
ax9.legend(fontsize=7, facecolor="#333", labelcolor="white")
ax9.set_ylabel("Count", color="#cccccc")

# 10. Context length by source boxplot
ax10 = fig.add_subplot(gs[3, 0])
src_list   = df["source"].unique()
box_data   = [df[df["source"]==s]["ctx_len"].values for s in src_list]
bp = ax10.boxplot(box_data, patch_artist=True,
                  boxprops=dict(facecolor="#37474f", color="#90caf9"),
                  medianprops=dict(color="#00e676"),
                  whiskerprops=dict(color="#90caf9"),
                  capprops=dict(color="#90caf9"),
                  flierprops=dict(markerfacecolor="#ff5252", markersize=3))
ax10.set_xticklabels(src_list, fontsize=8)
style_ax(ax10, "Context Length by Source (words)")
ax10.set_ylabel("Words", color="#cccccc")

# 11. Difficulty × Action heatmap
ax11   = fig.add_subplot(gs[3, 1])
heat   = pd.crosstab(df["difficulty"], df["action"])
heat   = heat.reindex([o for o in order if o in heat.index])
im     = ax11.imshow(heat.values, cmap="YlOrRd", aspect="auto")
ax11.set_xticks(range(len(heat.columns)))
ax11.set_yticks(range(len(heat.index)))
ax11.set_xticklabels(heat.columns, color="white", fontsize=8)
ax11.set_yticklabels(heat.index, color="white", fontsize=8)
for i in range(heat.values.shape[0]):
    for j in range(heat.values.shape[1]):
        ax11.text(j, i, heat.values[i,j], ha="center",
                  va="center", color="black", fontsize=8)
style_ax(ax11, "Difficulty × Action Heatmap")

# 12. Missing variables distribution
ax12 = fig.add_subplot(gs[3, 2])
nv   = df["num_missing"].value_counts().sort_index()
ax12.bar(nv.index.astype(str), nv.values, color="#f48fb1", edgecolor="#1e1e1e")
style_ax(ax12, "# Missing Variables Distribution")
ax12.set_xlabel("# Missing vars", color="#cccccc")
ax12.set_ylabel("Count", color="#cccccc")

plt.suptitle("Unified Dataset — Full EDA", color="white",
             fontsize=14, y=1.01)
plt.savefig("/content/unified_eda.png", dpi=150,
            bbox_inches="tight", facecolor="#1e1e1e")
plt.show()
print("Saved → /content/unified_eda.png")